In [117]:
import string
import pandas as pd

from collections import Counter
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, pipeline

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer


df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
ds = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  A

In [88]:
# Getting the freequency distribution of the answers
counts = df["answer"].value_counts()
print(counts)

# Getting the most freequent and least freequent answer
most_freequency = counts.max()
least_freequency = counts.min()

result = most_freequency + least_freequency
print(result)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
814


In [89]:
# Make a translation table ( a mapping of characters ordinals to replacement )
# Basically replace all punchuations with nothing
translator = str.maketrans("", "", string.punctuation)

all_words = set()

# Getting each prompt at a time and performing cleanup
for text in df["prompt"]:
    cleaned = text.lower().translate(translator)
    words = cleaned.split()

    all_words.update(words)

print(len(all_words))

859


In [90]:
# Getting the Row ID = 1
row = df[df["id"] == 1]
text = row["prompt"].iloc[0]

# Using the translator form the cell above
cleaned = text.lower().translate(translator)
words = cleaned.split()

# Setting the lsit for filtered words
filtered_words = [
    word for word in words
    if word not in ENGLISH_STOP_WORDS
]

print(filtered_words)
print(len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
13


In [91]:
combined_text = (
    df["prompt"] + " " +
    df["A"] + " " +
    df["B"] + " " + 
    df["C"] + " " + 
    df["D"] + " " + 
    df["E"] 
)

# Converting into term freequency matrix using sklearn's TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words = 'english')
x = vectorizer.fit_transform(combined_text)

print(x.shape[1])

2762


In [92]:
# Getting the row variable from cell 4
prompt_vec = vectorizer.transform(row["prompt"])
A_vec = vectorizer.transform(row["A"])

similarity = cosine_similarity(prompt_vec, A_vec)[0, 0]

print(round(similarity, 4))

0.272


In [93]:
options = ["A", "B", "C", "D", "E"]

correct = 0

for _, row in df.iterrows():
    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in options:
        option_vec = vectorizer.transform([row[option]])

        # Checking the similarity between given prompt and option
        similarities[option] = cosine_similarity(prompt_vec, option_vec)[0, 0]

    prediction = max(similarities, key=similarities.get)

    if prediction == row["answer"]:
        correct += 1

accuracy = correct / len(df) * 100

print(round(accuracy, 2))

13.55


In [94]:
# Mean Average Precision @ 3
def map3_score(actual, predictions):
    for i, prediction in enumerate(predictions, start=1):
        if prediction == actual:
            return 1/i
            
    return 0

print(map3_score("C", ["C", "A", "B"]))
print(map3_score("B", ["D", "B", "E"]))

1.0
0.5


In [95]:
# Getting counts from cell 2
top3 = counts.index.tolist()[:3]
print(top3)

scores = []

for answer in df["answer"]:
    scores.append(map3_score(answer, top3))

map3 = sum(scores) / len(scores)

print(round(map3, 4))

['B', 'C', 'A']
0.4213


In [96]:
# Combine prompt and all options into one document per row
combined_text = (
    df["prompt"] + " " +
    df["A"] + " " +
    df["B"] + " " +
    df["C"] + " " +
    df["D"] + " " +
    df["E"]
)

# Fit TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined_text)


# MAP@3 function
def mapk(actual, predicted, k=3):
    score = 0.0

    for act, preds in zip(actual, predicted):
        for i, p in enumerate(preds[:k]):
            if p == act:
                score += 1.0 / (i + 1)
                break

    return score / len(actual)


# Generate top-3 predictions
all_preds = []

for _, row in df.iterrows():

    prompt_vec = tfidf.transform([row["prompt"]])

    sims = {}

    for opt in options:

        opt_vec = tfidf.transform([row[opt]])

        sims[opt] = cosine_similarity(
            prompt_vec,
            opt_vec
        )[0][0]

    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    all_preds.append(ranked[:3])


# Compute MAP@3
score_q10 = mapk(
    df["answer"].tolist(),
    all_preds
)

print(round(score_q10, 4))

0.2962


In [97]:
# Create combined_text column
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    
    return example

ds = ds.map(combine)

# Length of row index 51
print(len(ds[51]["combined_text"]))

614


In [98]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions = True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [99]:
# Printing the exact vocabulary size hardcoded into the tokenizer
print("Vocabulary Size : ", tokenizer.vocab_size)

print("SEP Token : ", tokenizer.sep_token)
print("SEP Token ID : ", tokenizer.sep_token_id)

Vocabulary Size :  30522
SEP Token :  [SEP]
SEP Token ID :  102


In [100]:
# Tokenizing entire list for faster parallel processing
encoded = tokenizer(
    list(ds["prompt"]),
    padding = "max_length",
    truncation = True,
    max_length = 128,
    return_tensors = "pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


In [101]:
hidden_size = 768
num_heads = 12

head_dim = hidden_size // num_heads

print(head_dim)

64


In [102]:
# Loading a BERT model and tokenizing only the first prompt
text = ds[0]["prompt"]

inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


In [103]:
cls_embedding = outputs.last_hidden_state[0,0]
result = cls_embedding[:5].sum()

print(round(result.item(),4))

-1.2001


In [104]:
text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
fusion_index = tokens.index("fusion")

attention = outputs.attentions[-1][0,0]
score = attention[0,fusion_index]

print(tokens)
print(round(score.item(),4))

['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


In [105]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-V2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [106]:
prompt = ds[0]["prompt"]
option = ds[0]["B"]

embeddings = model.encode(
    [prompt, option],
    convert_to_tensor=True
)

similarity = util.cos_sim(
    embeddings[0],
    embeddings[1]
)

print(round(similarity.item(), 4))

0.7658


In [107]:
OPTIONS = ["A","B","C","D","E"]

def mapk(actual, predicted, k=3):
    score = 0

    for act,preds in zip(actual,predicted):
        for i,p in enumerate(preds[:k]):
            if p==act:
                score += 1/(i+1)
            
                break

    return score/len(actual)

mini_preds = []

for row in ds:

    texts = [row["prompt"]] + [row[o] for o in OPTIONS]

    embeddings = model.encode(
        texts,
        convert_to_tensor=True
    )

    prompt_emb = embeddings[0]

    sims = {}

    for i,opt in enumerate(OPTIONS):
        sims[opt] = util.cos_sim(
            prompt_emb,
            embeddings[i+1]
        ).item()

    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    mini_preds.append(ranked[:3])

mini_map = mapk(
    ds["answer"],
    mini_preds
)

print("MiniLM MAP@3 : ", mini_map)

MiniLM MAP@3 :  0.42308333333333487


In [110]:
count = 0

for actual, tfidf, mini in zip(ds["answer"], all_preds, mini_preds):
    if actual not in tfidf and actual in mini:
        count += 1

print(count)

502


In [111]:
classifier = pipeline(
    "zero-shot-classification"
)

sequence = ds[1]["prompt"]

labels = [
    ds[1]["A"],
    ds[1]["B"],
    ds[1]["C"]
]

result = classifier(
    sequence,
    candidate_labels=labels
)

print(result)
print(round(result["scores"][0],4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [112]:
result1 = classifier(sequence, candidate_labels=labels)

result2 = classifier(
    sequence,
    candidate_labels=labels,
    multi_label=True
)

sum1 = sum(result1["scores"])
sum2 = sum(result2["scores"])

print(abs(sum1-sum2))

0.9994903318565775


In [118]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [120]:
row = ds[0]

text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

B
